In [77]:
from transformers import AutoTokenizer, BertConfig, BertModel
from torch.utils.data import Dataset
import pandas as pd
import torch
from llm2vec import LLM2Vec

In [2]:
sel_cols = ['assetlongdescription_entity_llms', 'failurelocation_original', 
            'assetlongdescription_original', 'mode']
df = pd.read_csv('../processed/asset2item.csv')[sel_cols]
df.rename({'assetlongdescription_original': 'answers.text'}, axis=1, inplace=True)
df['answers.text'] = pd.Series(df['answers.text'], dtype="string")

In [4]:
df_train = df[df['mode']=='train']
df_val = df[df['mode']=='val']
df_test = df[df['mode']=='test']

In [6]:
chkpt_path = 'entity_masking_mlm_bert/checkpoint-14000'

In [52]:
tokenizer = AutoTokenizer.from_pretrained('google-bert/bert-base-uncased')
config = BertConfig.from_pretrained(chkpt_path, output_hidden_states=True)
model = BertModel.from_pretrained(chkpt_path, config=config)

Some weights of BertModel were not initialized from the model checkpoint at entity_masking_mlm_bert/checkpoint-14000 and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [41]:
class CustomDataset(Dataset):
    def __init__(self, df):
        self.df = df
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        item = {
            'text': self.df.iloc[idx]['answers.text'],
            'failure_locations': self.df.iloc[idx]['failurelocation_original'],
            'labels': self.df.iloc[idx].assetlongdescription_entity_llms
        }
        return item

In [42]:
ds_train = CustomDataset(df_train)
ds_val = CustomDataset(df_val)
ds_test = CustomDataset(df_test)

In [43]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [53]:
_ = model.to(device)

In [59]:
print(item['text'])

The equipment Accumulator - Pneumatic - Bladder Type, is categorized as Fixed Asset and has the following boundary: A Pneumatic Accumulator - Bladder Type in this database is comprised of:  - Tank - Bladder - Air Line Check Valve, if present - Gas Precharge Valve


## BERT embedding

In [73]:
def embed(model, text):
    return model(**input).last_hidden_state.mean(axis=1)

In [75]:
embedding = embed(model, ds_train[0])

In [76]:
embedding.shape

torch.Size([1, 768])

## LLM2Vec embedding

In [92]:
l2v = LLM2Vec.from_pretrained(
    "McGill-NLP/LLM2Vec-Mistral-7B-Instruct-v2-mntp",
    peft_model_name_or_path="/dccstor/dcpfactory/llm2vec/output/mlm/Mistral-7B-Instruct-v0.2",
    device_map="cuda" if torch.cuda.is_available() else "cpu",
    torch_dtype=torch.bfloat16,
)

Loading checkpoint shards: 100%|██████████████████| 3/3 [00:23<00:00,  7.88s/it]


In [96]:
instruction = (
    "Given an asset description, give me the failure locations:"
)
queries = [
    [instruction, ds_train[0]['text']],
]
q_reps = l2v.encode(queries)

Batches: 100%|████████████████████████████████████| 1/1 [00:00<00:00, 18.69it/s]


In [99]:
q_reps.shape

torch.Size([1, 4096])